# Task 7 — Pixel-based anomaly segmentation baselines

This notebook evaluates the pixel-based anomaly segmentation baselines required in **Task 7**.

The model used here is the pretrained **ERFNet** semantic segmentation network. Since ERFNet outputs dense pixel-wise class logits, anomaly scores can be computed directly from the logits using post-hoc uncertainty/confidence methods.

The evaluated methods are:

- **MaxLogit**: anomaly score = negative maximum class logit;
- **MSP**: anomaly score = one minus the maximum softmax probability;
- **Max Entropy**: anomaly score = entropy of the softmax distribution.

The output of this notebook is a compact table with **AUPRC** and **FPR95** for each dataset and each method.

## 1. Environment and paths

This notebook assumes that the repository and large files have already been prepared by the setup notebook.  
Only Google Drive is mounted here, and the project paths are defined.

In [1]:
# Environment prepared by 00_setup: here we only mount Google Drive and define paths.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "MaskArchitectureAnomaly_CourseProject"
EVAL_DIR = PROJECT_ROOT / "eval"
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"

# Make project modules importable if needed.
for p in (PROJECT_ROOT, PROJECT_ROOT / "eomt", EVAL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
        
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("EVAL_DIR exists:", EVAL_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT exists: True
EVAL_DIR exists: True


## 2. Load the anomaly validation datasets

The anomaly validation datasets are stored as a zip file in Google Drive.  
They are extracted temporarily to `/content` so that inference is faster during the Colab session.

In [2]:
from pathlib import Path
import zipfile

ANOMALY_ZIP = LARGE_FILES / "datasets" / "anomaly" / "Anomaly_Validation_Datasets.zip"
LOCAL_DATA_DIR = Path("/content/anomaly_data")

print("Zip exists:", ANOMALY_ZIP.exists())
if not LOCAL_DATA_DIR.exists():
    print("Extracting anomaly datasets temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Extraction completed.")
else:
    print("Datasets already extracted in this runtime.")

DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"
TRAINED_MODELS_DIR = PROJECT_ROOT / "trained_models"
ERFNET_WEIGHTS = TRAINED_MODELS_DIR / "erfnet_pretrained.pth"

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("TRAINED_MODELS_DIR exists:", TRAINED_MODELS_DIR.exists())
print("ERFNet weights exist:", ERFNET_WEIGHTS.exists())

Zip exists: True
Extracting anomaly datasets temporarily to /content...
Extraction completed.
DATA_ROOT exists: True
TRAINED_MODELS_DIR exists: True
ERFNet weights exist: True


## 3. Define datasets and post-hoc methods

The same anomaly validation datasets are used for all methods.  
The image extension is dataset-specific, so each dataset uses its own glob pattern.

In [3]:
import glob

# Dataset image patterns. These names are kept consistent with the project folder names.
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

# Pixel-based post-hoc anomaly scoring methods required for Task 7.
methods = ["maxlogit", "msp", "entropy"]

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")

FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


## 4. Sanity checks

Before running the full evaluation, we check that the evaluation script, ERFNet implementation and pretrained weights are available.


In [4]:
print("Evaluation script exists:", (EVAL_DIR / "evalAnomaly.py").exists())
print("ERFNet implementation exists:", (EVAL_DIR / "erfnet.py").exists())
print("ERFNet pretrained weights exist:", ERFNET_WEIGHTS.exists())

Evaluation script exists: True
ERFNet implementation exists: True
ERFNet pretrained weights exist: True


## 5. Run ERFNet anomaly evaluation

The original `evalAnomaly.py` script is used with minimal changes to the workflow.  
For each dataset and each method, the script is launched as a subprocess. The notebook parses the printed metrics and stores them in a structured table.

The full stdout of each run is saved in memory only for inspection, while the notebook prints a compact summary suitable for reporting.

In [5]:
import re
import subprocess
import pandas as pd
from datetime import datetime

results = []
full_logs = {}

AUPRC_RE = re.compile(r"AUPRC score:\s*([0-9.]+)")
FPR_RE = re.compile(r"FPR@TPR95:\s*([0-9.]+)")

start_time = datetime.now()
print("Evaluation started at:", start_time.strftime("%Y-%m-%d %H:%M:%S"))

for dataset_name, input_pattern in datasets.items():
    for method in methods:
        print(f"\nRunning {method:8s} on {dataset_name}")

        cmd = [
            "python", str(EVAL_DIR / "evalAnomaly.py"),
            "--input", str(input_pattern),
            "--loadDir", str(TRAINED_MODELS_DIR) + "/",
            "--loadWeights", "erfnet_pretrained.pth",
            "--method", method,
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        stdout = result.stdout
        stderr = result.stderr
        full_logs[(dataset_name, method)] = {"stdout": stdout, "stderr": stderr}

        if result.returncode != 0:
            print("  ERROR: evaluation script failed.")
            print(stderr)
            continue

        auprc_match = AUPRC_RE.search(stdout)
        fpr_match = FPR_RE.search(stdout)

        if auprc_match is None or fpr_match is None:
            print("  WARNING: metrics could not be parsed from stdout.")
            if stderr:
                print("  STDERR:", stderr[:1000])
            continue

        auprc = float(auprc_match.group(1))
        fpr95 = float(fpr_match.group(1))

        results.append({
            "model": "ERFNet",
            "dataset": dataset_name,
            "method": method,
            "AUPRC": auprc,
            "FPR95": fpr95,
        })

        print(f"  AUPRC = {auprc:.2f} | FPR95 = {fpr95:.2f}")

end_time = datetime.now()
print("\nEvaluation completed at:", end_time.strftime("%Y-%m-%d %H:%M:%S"))
print("Elapsed time:", end_time - start_time)

results_df = pd.DataFrame(results)
results_df

Evaluation started at: 2026-06-03 14:09:25

Running maxlogit on FS_LostFound_full
  AUPRC = 3.30 | FPR95 = 45.53

Running msp      on FS_LostFound_full
  AUPRC = 1.75 | FPR95 = 50.66

Running entropy  on FS_LostFound_full
  AUPRC = 2.58 | FPR95 = 50.21

Running maxlogit on fs_static
  AUPRC = 9.50 | FPR95 = 40.30

Running msp      on fs_static
  AUPRC = 7.47 | FPR95 = 41.84

Running entropy  on fs_static
  AUPRC = 8.84 | FPR95 = 41.55

Running maxlogit on RoadAnomaly
  AUPRC = 15.58 | FPR95 = 73.28

Running msp      on RoadAnomaly
  AUPRC = 12.42 | FPR95 = 82.58

Running entropy  on RoadAnomaly
  AUPRC = 12.67 | FPR95 = 82.76

Running maxlogit on RoadAnomaly21
  AUPRC = 38.30 | FPR95 = 59.37

Running msp      on RoadAnomaly21
  AUPRC = 29.08 | FPR95 = 62.56

Running entropy  on RoadAnomaly21
  AUPRC = 30.96 | FPR95 = 62.68

Running maxlogit on RoadObsticle21
  AUPRC = 4.63 | FPR95 = 48.44

Running msp      on RoadObsticle21
  AUPRC = 2.71 | FPR95 = 65.18

Running entropy  on RoadObstic

,model,dataset,method,AUPRC,FPR95
0,ERFNet,FS_LostFound_full,maxlogit,3.300986,45.532955
1,ERFNet,FS_LostFound_full,msp,1.747985,50.657505
2,ERFNet,FS_LostFound_full,entropy,2.582244,50.213916
3,ERFNet,fs_static,maxlogit,9.499818,40.302110
4,ERFNet,fs_static,msp,7.473543,41.837639
5,ERFNet,fs_static,entropy,8.838664,41.546897
6,ERFNet,RoadAnomaly,maxlogit,15.581526,73.279385
7,ERFNet,RoadAnomaly,msp,12.421447,82.583275
8,ERFNet,RoadAnomaly,entropy,12.668263,82.756523
9,ERFNet,RoadAnomaly21,maxlogit,38.296645,59.373450


## 6. Final result tables

In [7]:
# AUPRC table: higher is better.
auprc_table = results_df.pivot(index="dataset", columns="method", values="AUPRC")
auprc_table = auprc_table[["maxlogit", "msp", "entropy"]]
auprc_table["best_method"] = auprc_table[["maxlogit", "msp", "entropy"]].idxmax(axis=1)
auprc_table

method,maxlogit,msp,entropy,best_method
dataset,,,,
FS_LostFound_full,3.300986,1.747985,2.582244,maxlogit
RoadAnomaly,15.581526,12.421447,12.668263,maxlogit
RoadAnomaly21,38.296645,29.083959,30.956045,maxlogit
RoadObsticle21,4.630934,2.713296,3.048639,maxlogit
fs_static,9.499818,7.473543,8.838664,maxlogit


In [8]:
# FPR95 table: lower is better.
fpr95_table = results_df.pivot(index="dataset", columns="method", values="FPR95")
fpr95_table = fpr95_table[["maxlogit", "msp", "entropy"]]
fpr95_table["best_method"] = fpr95_table[["maxlogit", "msp", "entropy"]].idxmin(axis=1)
fpr95_table

method,maxlogit,msp,entropy,best_method
dataset,,,,
FS_LostFound_full,45.532955,50.657505,50.213916,maxlogit
RoadAnomaly,73.279385,82.583275,82.756523,maxlogit
RoadAnomaly21,59.373450,62.562897,62.675714,maxlogit
RoadObsticle21,48.437955,65.180470,65.876601,maxlogit
fs_static,40.302110,41.837639,41.546897,maxlogit


## 7. Save results

In [9]:
PROJECT_PARENT = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project"
OUTPUT_DIR = PROJECT_PARENT / "results" / "task7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = OUTPUT_DIR / "task7_erfnet_raw_results.csv"
auprc_csv = OUTPUT_DIR / "task7_erfnet_auprc_table.csv"
fpr95_csv = OUTPUT_DIR / "task7_erfnet_fpr95_table.csv"

results_df.to_csv(raw_csv, index=False)
auprc_table.to_csv(auprc_csv)
fpr95_table.to_csv(fpr95_csv)

print("Saved raw results to:", raw_csv)
print("Saved AUPRC table to:", auprc_csv)
print("Saved FPR95 table to:", fpr95_csv)

Saved raw results to: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task7/task7_erfnet_raw_results.csv
Saved AUPRC table to: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task7/task7_erfnet_auprc_table.csv
Saved FPR95 table to: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task7/task7_erfnet_fpr95_table.csv


In [10]:
excel_path = OUTPUT_DIR / "task7_erfnet_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="raw_results", index=False)
    auprc_table.to_excel(writer, sheet_name="AUPRC")
    fpr95_table.to_excel(writer, sheet_name="FPR95")

print("Saved Excel results to:", excel_path)

Saved Excel results to: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task7/task7_erfnet_results.xlsx
